In [1]:
import pandas as pd
import numpy as np
import os

print(f"Current directory: {os.getcwd()}")

df = pd.read_csv('../data/processed/texas_part_d_clean.csv', low_memory=False)

print(f"Shape: {df.shape}")
print(f"Unique specialties: {df['Prscrbr_Type'].nunique()}")
print(f"Unique providers: {df['Prscrbr_NPI'].nunique()}")

Current directory: /home/bri/medicare-fraud-detection/notebooks
Shape: (869717, 25)
Unique specialties: 97
Unique providers: 62301


In [2]:
# Calculate peer group benchmarks by specialty
specialty_benchmarks = df.groupby('Prscrbr_Type').agg(
    avg_cost_per_claim=('cost_per_claim', 'mean'),
    std_cost_per_claim=('cost_per_claim', 'std'),
    avg_days_per_claim=('days_per_claim', 'mean'),
    std_days_per_claim=('days_per_claim', 'std'),
    avg_tot_clms=('Tot_Clms', 'mean'),
    std_tot_clms=('Tot_Clms', 'std'),
    avg_brand_rate=('is_brand', 'mean'),
    provider_count=('Prscrbr_NPI', 'nunique')
).reset_index()

print(f"Benchmarks calculated for {len(specialty_benchmarks)} specialties")
print(f"\nTop 10 highest avg cost per claim specialties:")
print(specialty_benchmarks.nlargest(10, 'avg_cost_per_claim')[
    ['Prscrbr_Type', 'avg_cost_per_claim', 'provider_count']
])

Benchmarks calculated for 97 specialties

Top 10 highest avg cost per claim specialties:
                                         Prscrbr_Type  avg_cost_per_claim  \
34  Hematopoietic Cell Transplantation And Cellula...         1649.075770   
32                                         Hematology          554.428178   
22                                      Endocrinology          550.436840   
37                                 Infectious Disease          544.009466   
46                                   Medical Oncology          425.920318   
85                                       Rheumatology          424.516019   
82                                  Pulmonary Disease          387.826941   
16                       Critical Care (Intensivists)          360.850780   
33                                Hematology-Oncology          349.991461   
70                                         Pharmacist          325.774295   

    provider_count  
34              11  
32              25  


In [3]:
# Merge benchmarks onto main dataframe
df = df.merge(specialty_benchmarks, on='Prscrbr_Type', how='left')

# Calculate Z-scores for each provider row vs their specialty peer group
df['z_cost_per_claim'] = (
    (df['cost_per_claim'] - df['avg_cost_per_claim']) / df['std_cost_per_claim']
)

df['z_days_per_claim'] = (
    (df['days_per_claim'] - df['avg_days_per_claim']) / df['std_days_per_claim']
)

df['z_tot_clms'] = (
    (df['Tot_Clms'] - df['avg_tot_clms']) / df['std_tot_clms']
)

print("Z-scores calculated")
print(f"\nZ-score summary:")
print(df[['z_cost_per_claim', 'z_days_per_claim', 'z_tot_clms']].describe())

Z-scores calculated

Z-score summary:
       z_cost_per_claim  z_days_per_claim    z_tot_clms
count      8.697120e+05      8.697120e+05  8.697120e+05
mean       3.137227e-18      6.640464e-17 -1.058814e-17
std        9.999477e-01      9.999477e-01  9.999477e-01
min       -1.515061e+00     -4.606889e+00 -1.807561e+00
25%       -2.332093e-01     -8.206839e-01 -3.367987e-01
50%       -1.824978e-01      6.215563e-02 -1.633141e-01
75%       -1.384684e-01      8.705935e-01  6.722351e-02
max        8.786217e+01      1.471197e+01  2.925316e+02


In [4]:
# Flag providers exceeding 2 standard deviations above peer average
# Using 2 as threshold — standard in statistical fraud detection
df['flag_high_cost'] = (df['z_cost_per_claim'] > 2).astype(int)
df['flag_high_days'] = (df['z_days_per_claim'] > 2).astype(int)
df['flag_high_clms'] = (df['z_tot_clms'] > 2).astype(int)
df['flag_high_brand'] = (df['is_brand'] == 1).astype(int)

# Composite flag: flagged on 2 or more signals
df['flag_count'] = (df['flag_high_cost'] + 
                    df['flag_high_days'] + 
                    df['flag_high_clms'] +
                    df['flag_high_brand'])

df['multi_flag'] = (df['flag_count'] >= 2).astype(int)

print(f"Flag summary:")
print(f"  High cost flags: {df['flag_high_cost'].sum()}")
print(f"  High days flags: {df['flag_high_days'].sum()}")
print(f"  High claims flags: {df['flag_high_clms'].sum()}")
print(f"  Multi-signal flags: {df['multi_flag'].sum()}")
print(f"\nMulti-flagged rows as % of dataset: {df['multi_flag'].mean()*100:.2f}%")

Flag summary:
  High cost flags: 32836
  High days flags: 6400
  High claims flags: 20844
  Multi-signal flags: 35911

Multi-flagged rows as % of dataset: 4.13%


In [5]:
# Aggregate to provider level: one row per NPI
provider_risk = df.groupby(['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 
                             'Prscrbr_First_Name', 'Prscrbr_City',
                             'Prscrbr_Type']).agg(
    total_claims=('Tot_Clms', 'sum'),
    total_cost=('Tot_Drug_Cst', 'sum'),
    unique_drugs=('Gnrc_Name', 'nunique'),
    avg_z_cost=('z_cost_per_claim', 'mean'),
    avg_z_days=('z_days_per_claim', 'mean'),
    avg_z_clms=('z_tot_clms', 'mean'),
    brand_rate=('is_brand', 'mean'),
    flag_high_cost=('flag_high_cost', 'max'),
    flag_high_days=('flag_high_days', 'max'),
    flag_high_clms=('flag_high_clms', 'max'),
    total_flags=('flag_count', 'sum'),
    multi_flag_rows=('multi_flag', 'sum')
).reset_index()

# Composite risk score: weighted combination of z-scores
provider_risk['risk_score'] = (
    (provider_risk['avg_z_cost'] * 0.4) +
    (provider_risk['avg_z_days'] * 0.3) +
    (provider_risk['avg_z_clms'] * 0.3)
).round(4)

print(f"Total providers scored: {len(provider_risk)}")
print(f"\nTop 15 highest risk providers:")
provider_risk.nlargest(15, 'risk_score')[
    ['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_Type',
     'total_cost', 'risk_score', 'total_flags']
]

Total providers scored: 62301

Top 15 highest risk providers:


,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_Type,total_cost,risk_score,total_flags
32152,1518325638,Farrell,Nurse Practitioner,1048877.03,24.0599,2
39806,1639571391,Jian,Physician Assistant,1868050.29,20.2592,2
53794,1861597080,Mcbryde,Physician Assistant,2090964.49,14.6687,4
20298,1326224056,Mcgarry,Emergency Medicine,766779.87,13.4917,3
53408,1851856926,Olvera,Family Practice,29183344.00,12.6256,29
61643,1982940466,Mayberry,Physician Assistant,436628.37,10.2692,2
24797,1396763207,Beecherl,General Surgery,170907.70,9.4401,2
51634,1821746835,Sicalag,Nurse Practitioner,2124354.81,8.8516,2
29045,1467569822,Oliva,Internal Medicine,50570752.74,8.4129,26
17668,1285184630,Shaw,Nurse Practitioner,1781537.60,8.3321,3
